# Week 4 Day 5 - Sidekick

これは、この1週間全体が積み上げてきた、最終的なプロジェクトです。Sidekickは、タスクと成功の定義を与えることができる、パーソナルな協働作業者です。本物のブラウザ、filesystem、Web検索などを使いながら作業を続け、あなたの基準を満たすか、あなたに何かを尋ねる必要が出るまで働き続けます。

これは、「はしごではなくスタック」という考え方が実を結ぶ場所でもあります。Sidekickのworkerは、レイヤー3の単一のcreate_agentですが、フレームワークにすべてを任せるのではなく、自分自身のループでそれを包み、必要に応じてレイヤー1のツールにも、外部のMCPサーバーにも自由に手を伸ばします。作業の各部分に応じて、適切な高さ（altitude）を自分で選ぶのです。

## どのように構築されているか

workerは、すべてのツールと、一連のミドルウェアを備えた1つの`create_agent`です。

- `TodoListMiddleware`は、作業を進めながら更新され続ける計画をworkerに与え、それをUI上に表示します。
- `PIIMiddleware`は、入力からメールアドレスを伏せ字にし、ブラウザが目にするクレジットカード番号さえも消し去ります。
- `ModelCallLimitMiddleware`は、1回の実行を30回のモデル呼び出しに制限し、迷走したagentが際限なくコストを消費しないようにします。
- `HumanInTheLoopMiddleware`は、プッシュ通知が送られる前にあなたの承認を待って一旦停止し、`request_human_help`ツールの動作を支えます。これによってagentは、ログインやcaptcha、二段階認証のプロンプトなどのために、ブラウザの操作をあなたに引き渡すことができます。
- 独自の`TolerateToolErrors`は、ツールの失敗をモデルに返し、回復できるようにします。

workerの周りには、私たち自身が書いた小さなループを実行します。

1. workerがタスクに取り組みます。
2. evaluator（structured outputを持つただのモデル）が、workerのツール呼び出しを証拠として、あなたの成功基準に対して回答を検証します。
3. 基準が満たされていれば、あるいはworkerが質問を持っている場合は、そこで止まり、結果を表示します。そうでなければ、フィードバックをworkerに返し、もう一度試させます。

裏側では、もう1つのことが起きています。ブラウザとfilesystemのMCPサーバーは、Sidekickが存在する間ずっと開いたままの永続的なセッションとして動作するので、ブラウザはツール呼び出しの間もその状態を保ち続け、タスクの途中で人間に操作を引き渡すことさえできます。これらのセッションは、`sidekick_tools.py`内の小さなバックグラウンドタスクの中で動作しています。これは、stdioのtransportが同じタスクから開かれ、閉じられなければならないためです。ありがたい副産物として、`cleanup()`は本当にブラウザを閉じてくれます。

これを3つのステップで組み立てていきます。まずはevaluatorのない最もシンプルなバージョン、次にミドルウェアを使ったhuman-in-the-loop、そして最後にプロジェクトのモジュールを使った完全なSidekickです。

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">実行する前に</h2>
            <span style="color:#ff7800;">このラボは<code>OPENAI_API_KEY</code>、<code>SERPER_API_KEY</code>、そしてPushoverのキーを使い、完全版のSidekickはNodeとnpxを通じて、ウィンドウを表示するブラウザ（headed browser）を起動します。ブラウザが作業している間、画面が占有されます。大きなタスクでは数分かかることもあります。プロジェクトファイルの<code>sidekick.py</code>、<code>sidekick_tools.py</code>、<code>app.py</code>は、このノートブックの隣にあります。
            </span>
        </td>
    </tr>
</table>

## Windows向けの1つの調整

Day 3と同じです。WindowsのJupyterの中からstdioのMCPサーバーを起動すると、カーネルのstderrに実際のファイルディスクリプタがなく、起動に失敗します。以下のセルは、サーバーのエラーログをリダイレクトして、すべてが正しく動作するようにします。MacとLinuxでは何も行いません。

In [ ]:
import sys

if sys.platform == "win32":
    import subprocess
    from functools import partial
    import langchain_mcp_adapters.sessions as mcp_sessions

    mcp_sessions.stdio_client = partial(mcp_sessions.stdio_client, errlog=subprocess.DEVNULL)
    print("Applied the Windows adjustment")
else:
    print("Not Windows, so nothing to do here")

In [ ]:
# まずはインポートと環境設定

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

from sidekick_tools import search, send_push_notification, wikipedia_lookup
from sidekick import Sidekick

load_dotenv(override=True)

## ステップ1: evaluatorのない、最もシンプルなSidekick

最も単純な形では、Sidekickは、いくつかのツールとメモリを持つcreate_agentと、1ターンの会話を実行する小さな関数から成ります。ツールはおなじみのものです。今週すでに使った、既製のSerper検索とWikipediaツール、そして自分たちのプッシュ通知ツールです。ここにはevaluatorはないので、workerは単に最善を尽くして答えるだけです。多くのタスクにおいては、このシンプルなバージョンだけで十分であり、これ以降に出てくるものはすべて、この上にレイヤーを重ねていくだけです。

In [ ]:
simple_worker = create_agent(
    model="openai:gpt-5.4-mini",
    tools=[search, send_push_notification, wikipedia_lookup],
    system_prompt="You are Sidekick, a helpful personal assistant. Use your tools to complete the task.",
    checkpointer=InMemorySaver(),
)

simple_worker

In [ ]:
async def ask(worker, message):
    config = {"configurable": {"thread_id": "simple-sidekick"}}
    result = await worker.ainvoke({"messages": [{"role": "user", "content": message}]}, config=config)
    return result["messages"][-1].content

reply = await ask(simple_worker, "Search for who won the Nobel Prize in Physics in 2023 and send a push notification with a short summary.")
print(reply)

## ステップ2: ミドルウェアによるhuman-in-the-loop

LangChainは、agentが一旦停止してユーザーからのフィードバックを収集できるようにする機能をサポートしています。`HumanInTheLoopMiddleware`に、どのツールで一旦停止すべきかを伝えると、agentはそれらを実行する代わりに停止し、私たちの判断を待ちます。

agentがそのようなツールに到達すると、実行は一旦停止し、保留中のアクションを記述するinterruptが返されます。そこで、私たちはそれを承認したり、編集したり、拒否したり、応答したりしてから、処理を再開できます。

In [ ]:
from langchain_core.tools import tool

@tool
def book_meeting(person: str, day: str) -> str:
    """Book a meeting with a person on a given day."""
    return f"Meeting booked with {person} on {day}."

approval_agent = create_agent(
    model="openai:gpt-5.4-mini",
    tools=[book_meeting],
    system_prompt="You are a scheduling assistant. Use the book_meeting tool.",
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"book_meeting": True})],
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "approval-demo"}}
result = await approval_agent.ainvoke(
    {"messages": [{"role": "user", "content": "Book a meeting with Sam on Friday."}]}, config
)

interrupt = result["__interrupt__"][0]
print("The agent paused and is asking for approval:")
print(interrupt.value["action_requests"][0]["description"])

agentは待機しています。私たちがそのアクションを承認すると、実行はまさに停止した場所から続行します。

このことを覚えておいてください。完全版のSidekickは、この同じミドルウェアを使ってプッシュ通知にゲートをかけ、また`request_human_help`ツールを動かしています。これによってagentは、あなたに、たとえばサイトへのログインといった何かを、agentのブラウザウィンドウの中で行うよう頼み、それが終わったら処理を続けることができます。

In [ ]:
resumed = await approval_agent.ainvoke(Command(resume={"decisions": [{"type": "approve"}]}), config)
print(resumed["messages"][-1].content)

## 応答の仕方を選ぶ

先ほどの予約は承認しましたが、承認はあくまで選択肢の1つに過ぎません。resumeは、一旦停止した各ツール呼び出しに対して1つの判断を持ち、それぞれの判断は次の3種類のいずれかになります。

- **承認（Approve）**は、提案された通りにツールを実行します: `{"type": "approve"}`。
- **編集（Edit）**は、先に変更した引数でツールを実行します: `{"type": "edit", "edited_action": {"name": "book_meeting", "args": {"person": "Sam", "day": "Monday"}}}`。
- **拒否（Reject）**はツールの実行をスキップし、モデルに別の方法を試せるようメモを返します: `{"type": "reject", "message": "No meetings on Fridays"}`。

そのため、予約を承認する代わりに断る場合は、次のように再開します。

```python
await approval_agent.ainvoke(
    Command(resume={"decisions": [{"type": "reject", "message": "No meetings on Fridays"}]}), config
)
```

これらのうちどれをそのツールに許可するかは、ミドルウェアを設定するときに決めます。`interrupt_on={"book_meeting": True}`は3つすべてを許可しますが、`{"book_meeting": {"allowed_decisions": ["approve", "reject"]}}`とすれば、承認か拒否のみを提供し、編集はできないようにできます。Sidekickは単純な承認だけに絞っていますが、編集や拒否も、resumeのその1行を変えるだけで使えます。

In [ ]:
config = {"configurable": {"thread_id": "rejection-demo2"}}
result = await approval_agent.ainvoke(
    {"messages": [{"role": "user", "content": "Book a meeting with Sam on Friday."}]}, config
)

interrupt = result["__interrupt__"][0]
print("The agent paused and is asking for approval:")
print(interrupt.value["action_requests"][0]["description"])

In [ ]:
resumed = await approval_agent.ainvoke(Command(resume={"decisions": [{"type": "reject", "message": "Rejected - no meetings on Fridays"}]}), config)
print(resumed["messages"][-1].content)

## ステップ3: 完全版のSidekick

完全版は`sidekick.py`と`sidekick_tools.py`にあります。workerは、フルセットのツールキットを持っています。永続的なMCPセッションによる、ウィンドウ表示のブラウザとsandboxのfilesystem、それに加えてWeb検索、Wikipedia、プッシュ通知、そして`request_human_help`です。このラボの最初に紹介したミドルウェアのスタックがすべて組み込まれており、evaluatorのループは、workerのツール呼び出しを証拠として使いながら、それぞれの答えをあなたの成功基準に照らして検証し、必要に応じてworkerを再度試させるために送り返します。

これは、スタック全体を1つのオブジェクトにまとめたものです。workerはレイヤー3のcreate_agentであり、そのツールはレイヤー1の`@tool`関数とMCPサーバーが組み合わさったもので、そのメモリはレイヤー2のcheckpointerであり、それを取り囲むループは自分自身で書いたコードです。それぞれの部分に、適切な高さ（altitude）を自分で選んだのです。

さあ、1つ動かしてみて、まずは簡単なブラウザの用事から始めましょう。ウィンドウが開き、自分で操作していく様子を見てみてください。Hacker Newsのトップストーリーは1日の中で変わっていくので、ここに示されているものとは違う見出しが表示されるはずです。

In [ ]:
sidekick = Sidekick()
await sidekick.setup()
print(f"Sidekick ready with {len(sidekick.tools)} tools")

In [ ]:
sidekick.tools

In [ ]:
from langchain_core.runnables.graph_mermaid import draw_mermaid_png
from IPython.display import Image
mm = sidekick.worker.get_graph().draw_mermaid().replace("[", " ").replace("]", " ")
Image(draw_mermaid_png(mm, output_file_path="graph.png"))

In [ ]:
history = await sidekick.run_turn(
    message="Go to Hacker News at news.ycombinator.com and tell me the title of the current top story.",
    success_criteria="The reply names a specific story currently on the Hacker News front page.",
    history=[],
)
for entry in history:
    print(f"[{entry['role']}] {entry['content'][:200]}\n")

## 最後の締め: 本物の用事

さて、この1週間にふさわしいタスクです。最良のフライトを見つけてもらいましょう。これには計画、ある程度まとまった実際のブラウジング、ファイル、そしてプッシュ通知が必要になり、最後にSidekickがあなたの許可を求めるところで終わります。`sidekick.py`の中で気付いてほしい細かな点があります。workerのsystem promptには、Google Flightsが自然言語のクエリをそのままURLに受け付けるという実用的なトリックが仕込まれていて、これがagentに専門的な能力を与えています。

作業している間、ブラウザのウィンドウを見て、計画も覗いてみてください。workerは`write_todos`ツールを通じてtodoリストを維持し、Sidekickはそれを`sidekick.todos`として公開しています。

In [ ]:
flight_task = """Find me the best round-trip flight from New York to London, leaving about a month from now
and returning a week later. I care about price first, then total journey time, and I would rather avoid
itineraries with two or more stops. Write your recommendation with the top three options to flights.md,
then send me a push notification with the price of your top pick."""

flight_criteria = "flights.md is written with three specific options including airline, times and price, plus a clear recommendation, and a push notification was sent with the recommended price."

history = await sidekick.run_turn(flight_task, flight_criteria, history)
print(history[-1]["content"])

Sidekickは作業を終え、今は一旦停止して、通知を送る許可を求めています。その計画がその経緯を物語っています。

In [ ]:
for todo in sidekick.todos:
    print(f"[{todo['status']}] {todo['content']}")

承認すると、通知があなたの携帯に届き、evaluatorはツール呼び出しを証拠として、基準が満たされたことを確認します。

In [ ]:
if sidekick.paused:
    history = await sidekick.resume(history)
for entry in history[-2:]:
    print(f"[{entry['role']}] {entry['content']}\n")

## Gradioアプリ

`app.py`は、これらすべてをチャットインターフェースの中に包み込みます。リクエストを入力するボックス、成功基準を入力するボックス、作業中にチャットの横でリアルタイムに更新されるSidekickの計画、そして一旦停止して判断を求めるときに現れるApproveボタンです。永続的なMCPセッションがきれいにシャットダウンされるので、Resetは今では本当にブラウザを閉じてくれます。

見た目や操作感は`styles.py`の中に、テーマ、CSSの定数、JSの定数として収められています。Gradio 6についての注意点として、これらは古いGradioのように`gr.Blocks()`に渡すのではなく、`launch()`に渡されます。ターミナルから`uv run app.py`でアプリを実行するか、まさに同じインターフェースをここで直接起動することもできます。

In [ ]:
from app import ui, LAUNCH_STYLE

ui.launch(**LAUNCH_STYLE)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00cc00;">おめでとうございます</h2>
            <span style="color:#00cc00;">あなたはSidekickを作り上げ、それによってスタック全体を旅してきました。レイヤー1の構成要素（building blocks）、LangGraphによるオーケストレーション、レイヤー3のcreate_agent、Deep Agentsのハーネス、そして今、それらすべてを組み合わせた本物のプロジェクトに、planning、guardrails、人間による承認のためのミドルウェアを添えて。これはかなりの量の能力であり、あなたはその一つ一つのレイヤーを理解しています。今週は本当に見事な仕事でした。
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">演習</h2>
            <span style="color:#ff7800;">Sidekickを本当に自分のものにしてみましょう。まずは身近な用事から始めてみてください。フライトのタスクのExpediaバージョンを試し、まずSidekickに自分のExpediaアカウントへログインさせてみましょう。ログインページに到達すると、agentは<code>request_human_help</code>を呼び出して一旦停止し、あなたがそのブラウザウィンドウでログインするのを待ち、その後検索を続けます。旅行サイトは自動化に対して防御策を取っているので、時折ブロックされることも想定しておいてください。それも、実世界のagentと向き合う作業の一部です。さらに進めるなら、<code>sidekick.py</code>の中でfilesystemの書き込みツールを承認ミドルウェアの背後に置き、自分自身のツールを追加し、今週本当に片付けたいタスクを、明確な成功基準とともにSidekickに与えてみましょう。
            </span>
        </td>
    </tr>
</table>